In [1]:
from datasets import load_dataset
import math

/home/achiket/Documents/udemy/full-stack-ai-engineer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, Dataset

stream = load_dataset(
    "Lichess/chess-position-evaluations",
    split="train",
    streaming=True
)

dataset = Dataset.from_list(list(stream.take(2_00_000)))

print(len(dataset))

200000


In [3]:
 
def win_percent_from_eval(cp, mate):
    """
    Converts a centipawn or mate score, from the perspective of the side
    to move in the FEN, into an estimated win percentage. Uses the same
    logistic shape lichess uses for their accuracy calculation.
    """
    if mate is not None:
        return 100.0 if mate > 0 else 0.0
 
    return 50 + 50 * (2 / (1 + math.exp(-0.00368208 * cp)) - 1)
 
 
def normalize_fen(fen):
    """
    The evaluations dataset stores FEN without halfmove clock or fullmove
    number. If joining against a dataset that includes the full six field
    FEN, strip it down to the first four fields to match.
    """
    parts = fen.split(" ")
    return " ".join(parts[:4])
 
 
def build_eval_lookup(dataset):
    """
    Builds a dict from normalized FEN to the best available eval row.
    When a FEN appears more than once, keeps the entry with the highest
    search depth since that is the more reliable evaluation.
    """
    lookup = {}
 
    for row in dataset:
        key = normalize_fen(row["fen"])
        existing = lookup.get(key)
 
        if existing is None or row["depth"] > existing["depth"]:
            lookup[key] = {
                "depth": row["depth"],
                "knodes": row["knodes"],
                "cp": row["cp"],
                "mate": row["mate"],
                "line": row["line"],
                "win_percent": win_percent_from_eval(row["cp"], row["mate"]),
            }
 
    return lookup
 
 
def summarize(lookup):
    depths = [v["depth"] for v in lookup.values()]
    mates = sum(1 for v in lookup.values() if v["mate"] is not None)
 
    print("unique normalized positions:", len(lookup))
    print("duplicate fens collapsed:", 200000 - len(lookup))
    print("min depth:", min(depths))
    print("max depth:", max(depths))
    print("average depth:", sum(depths) / len(depths))
    print("positions with certain mate:", mates)
 
 
def filter_by_min_depth(lookup, min_depth):
    filtered = {k: v for k, v in lookup.items() if v["depth"] >= min_depth}
    print("positions surviving depth >=", min_depth, ":", len(filtered))
    return filtered

In [4]:
lookup = build_eval_lookup(dataset)
summarize(lookup)

filtered = filter_by_min_depth(lookup, min_depth=15)

sample_key = next(iter(filtered))
print("example entry from filtered set")
print(sample_key, filtered[sample_key])

unique normalized positions: 28824
duplicate fens collapsed: 171176
min depth: 1
max depth: 245
average depth: 55.989002220371916
positions with certain mate: 5631
positions surviving depth >= 15 : 28656
example entry from filtered set
7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - - {'depth': 46, 'knodes': 4189972, 'cp': 69, 'mate': None, 'line': 'f7g7 e6e2 h8d8 e2d2 b7b5 c4b3 g7f6 d1e1 a6a5 a2a3', 'win_percent': 56.317641764335946}


In [12]:
!uv add chess


Resolved 131 packages in 12.03s                                      
   Building chess==1.11.2                                              ⠋ Preparing packages... (0/0)                                                   
   Building chess==1.11.2                                      
   Building chess==1.11.2                                      
   Building chess==1.11.2                                      
   Building chess==1.11.2                                      
      Built chess==1.11.2                                      
Prepared 1 package in 937ms                                              
Installed 1 package in 1ms                                  
 + chess==1.11.2


In [6]:
"""
Trains a classifier on the 200000 row Lichess evaluations dataset only.

The dataset gives the engine's own best line per position but does not
include human played moves, so a full blunder versus brilliant move
quality classifier cannot be trained from this alone. What it does give
is a real signal: for every position, the first move of the line field
is the engine's preferred move. This script trains a binary classifier
on that signal, positive examples are the engine preferred move, negative
examples are other legal moves sampled at the same position.

This produces the same feature pipeline the full move quality classifier
will need later, trained on data available right now, with no live
Stockfish calls required.
"""

import random
import chess
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

PULL_SIZE = 200000
NEGATIVES_PER_POSITION = 2
RANDOM_SEED = 42


def normalize_fen(fen):
    parts = fen.split(" ")
    return " ".join(parts[:4])


def to_full_fen(short_fen):
    """
    The dataset FEN omits halfmove clock and fullmove number. python-chess
    requires the full six field FEN to construct a board, so pad with
    default values, this does not affect legal move generation.
    """
    return f"{short_fen} 0 1"


def square_features(square):
    """
    Decomposes a square index into file, rank, and distance from center,
    which are meaningful spatial features, instead of feeding the raw
    0 to 63 index into the model as if it were an ordinal quantity.
    """
    file_idx = chess.square_file(square)
    rank_idx = chess.square_rank(square)
    center_distance = abs(file_idx - 3.5) + abs(rank_idx - 3.5)
    return file_idx, rank_idx, center_distance


def extract_features(board, move):
    is_capture = board.is_capture(move)
    is_check = board.gives_check(move)

    moving_piece = board.piece_at(move.from_square)
    piece_type = moving_piece.piece_type if moving_piece else 0
    piece_value = {
        chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
        chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
    }.get(piece_type, 0)

    captured_piece = board.piece_at(move.to_square)
    captured_value = {
        chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
        chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
    }.get(captured_piece.piece_type, 0) if captured_piece else 0

    material_balance = sum(
        len(board.pieces(pt, chess.WHITE)) - len(board.pieces(pt, chess.BLACK))
        for pt in [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN]
    )

    board_copy = board.copy()
    board_copy.push(move)
    gives_check_after = board_copy.is_check()

    from_file, from_rank, from_center_dist = square_features(move.from_square)
    to_file, to_rank, to_center_dist = square_features(move.to_square)

    return {
        "is_capture": int(is_capture),
        "is_check": int(is_check),
        "gives_check_after": int(gives_check_after),
        "moving_piece_value": piece_value,
        "captured_value": captured_value,
        "material_balance": material_balance,
        "is_promotion": int(move.promotion is not None),
        "from_file": from_file,
        "from_rank": from_rank,
        "from_center_dist": from_center_dist,
        "to_file": to_file,
        "to_rank": to_rank,
        "to_center_dist": to_center_dist,
        "num_legal_moves": board.legal_moves.count(),
    }


def build_training_rows(dataset):
    rows = []
    skipped = 0

    for record in dataset:
        short_fen = normalize_fen(record["fen"])
        full_fen = to_full_fen(short_fen)

        if not record["line"]:
            skipped += 1
            continue

        try:
            board = chess.Board(full_fen)
        except ValueError:
            skipped += 1
            continue

        best_move_uci = record["line"].split(" ")[0]

        try:
            best_move = chess.Move.from_uci(best_move_uci)
        except ValueError:
            skipped += 1
            continue

        if best_move not in board.legal_moves:
            skipped += 1
            continue

        positive_features = extract_features(board, best_move)
        rows.append((positive_features, 1, short_fen))

        legal_moves = [m for m in board.legal_moves if m != best_move]
        if not legal_moves:
            continue

        sample_count = min(NEGATIVES_PER_POSITION, len(legal_moves))
        for negative_move in random.sample(legal_moves, sample_count):
            negative_features = extract_features(board, negative_move)
            rows.append((negative_features, 0, short_fen))

    print("rows skipped due to bad fen or move parsing:", skipped)
    return rows


def train_and_evaluate(rows):
    from sklearn.model_selection import GroupShuffleSplit

    feature_keys = list(rows[0][0].keys())
    X = [[row[0][k] for k in feature_keys] for row in rows]
    y = [row[1] for row in rows]
    groups = [row[2] for row in rows]

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
    train_idx, test_idx = next(splitter.split(X, y, groups))

    X_train = [X[i] for i in train_idx]
    X_test = [X[i] for i in test_idx]
    y_train = [y[i] for i in train_idx]
    y_test = [y[i] for i in test_idx]

    train_fens = set(groups[i] for i in train_idx)
    test_fens = set(groups[i] for i in test_idx)
    print("overlapping fens between train and test:", len(train_fens & test_fens))

    model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    print(classification_report(y_test, predictions, target_names=["not_engine_best", "engine_best"]))

    importances = sorted(zip(feature_keys, model.feature_importances_), key=lambda x: -x[1])
    print("feature importances")
    for name, score in importances:
        print(name, round(score, 4))

    return model, feature_keys


if __name__ == "__main__":
    random.seed(RANDOM_SEED) 
    rows = build_training_rows(dataset)

    print("total training rows built:", len(rows))
    print("positive examples:", sum(1 for r in rows if r[1] == 1))
    print("negative examples:", sum(1 for r in rows if r[1] == 0))

    model, feature_keys = train_and_evaluate(rows)

rows skipped due to bad fen or move parsing: 0
total training rows built: 597369
positive examples: 200000
negative examples: 397369
overlapping fens between train and test: 0
                 precision    recall  f1-score   support

not_engine_best       0.71      0.95      0.81     77536
    engine_best       0.69      0.22      0.33     39023

       accuracy                           0.71    116559
      macro avg       0.70      0.58      0.57    116559
   weighted avg       0.70      0.71      0.65    116559

feature importances
captured_value 0.172
to_center_dist 0.1082
moving_piece_value 0.1068
from_center_dist 0.1015
to_file 0.0944
num_legal_moves 0.0854
is_capture 0.0751
from_file 0.0744
to_rank 0.0552
from_rank 0.0505
material_balance 0.0352
is_check 0.0219
gives_check_after 0.0189
is_promotion 0.0006
